#### Implement the current scoring formula in puer Python (no Django dependency) so tests run standalone

In [ ]:
def compute_scores(categories, category_avgs):
    total_w = sum(c["weight"] for c in categories)

    overall = 0.0
    cat_scores = []

    for cat in categories:
        avg = category_avgs.get(cat["name"], 0.0)

        cat_scores.append({
            "category": cat["name"],
            "avg": avg,
            "pct": (avg / 5.0) * 100.0
        })

        overall += (
            (avg / 5.0)
            * (cat["weight"] / total_w)
            * 100.0
        )

    return cat_scores, overall

In [ ]:
categories = [
    {"name": "Demand", "weight": 0.4},
    {"name": "Conversion", "weight": 0.4},
    {"name": "Delivery", "weight": 0.2},
]

category_avgs = {
    "Demand": 4.5,
    "Conversion": 2.0,
    "Delivery": 3.0,
}

cat_scores, overall = compute_scores(categories, category_avgs)

assert round(overall, 1) == 64.0

#### Assertions for overall score and all three category scores for each edge case

In [ ]:
# If our pure Python function returns....
cat_scores, overall = compute_scores(categories, category_avgs)

results = {
    row["category"]: row
    for row in cat_scores
}

In [ ]:
# Then the assertions for all 5 edge cases would be...
# ============================================================
# Case 1 — Strong Demand, Weak Conversion, Mid Delivery
# ============================================================

category_avgs = {
    "Demand": 4.5,
    "Conversion": 2.0,
    "Delivery": 3.0,
}

cat_scores, overall = compute_scores(categories, category_avgs)

results = {row["category"]: row for row in cat_scores}

assert round(results["Demand"]["avg"], 2) == 4.50
assert round(results["Conversion"]["avg"], 2) == 2.00
assert round(results["Delivery"]["avg"], 2) == 3.00
assert round(overall, 1) == 64.0


# ============================================================
# Case 2 — Weak Demand, Strong Conversion, Mid Delivery
# ============================================================

category_avgs = {
    "Demand": 2.0,
    "Conversion": 4.5,
    "Delivery": 3.0,
}

cat_scores, overall = compute_scores(categories, category_avgs)

results = {row["category"]: row for row in cat_scores}

assert round(results["Demand"]["avg"], 2) == 2.00
assert round(results["Conversion"]["avg"], 2) == 4.50
assert round(results["Delivery"]["avg"], 2) == 3.00
assert round(overall, 1) == 64.0


# ============================================================
# Case 3 — All Strong
# ============================================================

category_avgs = {
    "Demand": 4.5,
    "Conversion": 4.5,
    "Delivery": 4.5,
}

cat_scores, overall = compute_scores(categories, category_avgs)

results = {row["category"]: row for row in cat_scores}

assert round(results["Demand"]["avg"], 2) == 4.50
assert round(results["Conversion"]["avg"], 2) == 4.50
assert round(results["Delivery"]["avg"], 2) == 4.50
assert round(overall, 1) == 90.0


# ============================================================
# Case 4 — All Weak
# ============================================================

category_avgs = {
    "Demand": 1.5,
    "Conversion": 1.5,
    "Delivery": 1.5,
}

cat_scores, overall = compute_scores(categories, category_avgs)

results = {row["category"]: row for row in cat_scores}

assert round(results["Demand"]["avg"], 2) == 1.50
assert round(results["Conversion"]["avg"], 2) == 1.50
assert round(results["Delivery"]["avg"], 2) == 1.50
assert round(overall, 1) == 30.0


# ============================================================
# Case 5 — Mid Demand, Mid Conversion, Weak Delivery
# ============================================================

category_avgs = {
    "Demand": 3.0,
    "Conversion": 3.0,
    "Delivery": 2.0,
}

cat_scores, overall = compute_scores(categories, category_avgs)

results = {row["category"]: row for row in cat_scores}

assert round(results["Demand"]["avg"], 2) == 3.00
assert round(results["Conversion"]["avg"], 2) == 3.00
assert round(results["Delivery"]["avg"], 2) == 2.00
assert round(overall, 1) == 56.0

In [ ]:
# Prepare these assertions into proper pytest format by PARAMETERIZING them
import pytest

@pytest.mark.parametrize(
    "demand,conversion,delivery,expected_overall",
    [
        (4.5, 2.0, 3.0, 64.0),
        (2.0, 4.5, 3.0, 64.0),
        (4.5, 4.5, 4.5, 90.0),
        (1.5, 1.5, 1.5, 30.0),
        (3.0, 3.0, 2.0, 56.0),
    ],
)
def test_scoring_formula(
    demand,
    conversion,
    delivery,
    expected_overall,
):
    category_avgs = {
        "Demand": demand,
        "Conversion": conversion,
        "Delivery": delivery,
    }

    cat_scores, overall = compute_scores(categories, category_avgs)

    results = {row["category"]: row for row in cat_scores}

    assert results["Demand"]["avg"] == demand
    assert results["Conversion"]["avg"] == conversion
    assert results["Delivery"]["avg"] == delivery
    assert round(overall, 1) == expected_overall

**Notes:** These tests validate the current GTM scoring formula using five edge-case score vectors. For each case, the test passes predefined Demand, Conversion, and Delivery averages into the scoring function, computes the resulting category and overall scores, and uses assertions to verify that the outputs match the expected values. The **@pytest.mark.parametrize** decorator allows the same test logic to run automatically across all five scenarios, making it easy to detect unintended changes to the scoring model.

#### Run pytest — confirm all 6 cases pass against the current formula

In [ ]:
# Create the test file
%%writefile test_scoring.py

# Full contents of test_scoring.py here
# ── scoring formula ──────────────────────────────────────────────────────────
def compute_scores(categories, category_avgs):
    total_w = sum(c["weight"] for c in categories)

    overall = 0.0
    cat_scores = []

    for cat in categories:
        avg = category_avgs.get(cat["name"], 0.0)

        cat_scores.append({
            "category": cat["name"],
            "avg": avg,
            "pct": (avg / 5.0) * 100.0
        })

        overall += (
            (avg / 5.0)
            * (cat["weight"] / total_w)
            * 100.0
        )

    return cat_scores, overall


# ── shared fixture ────────────────────────────────────────────────────────────
CATEGORIES = [
    {"name": "Demand",     "weight": 0.4},
    {"name": "Conversion", "weight": 0.4},
    {"name": "Delivery",   "weight": 0.2},
]

# ── parametrized tests ────────────────────────────────────────────────────────
import pytest

@pytest.mark.parametrize(
    "case_label, demand, conversion, delivery, expected_overall",
    [
        ("Case 1 — Strong Demand, Weak Conversion, Mid Delivery",  4.5, 2.0, 3.0, 64.0),
        ("Case 2 — Weak Demand, Strong Conversion, Mid Delivery",  2.0, 4.5, 3.0, 64.0),
        ("Case 3 — All Strong",                                    4.5, 4.5, 4.5, 90.0),
        ("Case 4 — All Weak",                                      1.5, 1.5, 1.5, 30.0),
        ("Case 5 — Mid Demand, Mid Conversion, Weak Delivery",     3.0, 3.0, 2.0, 56.0),
        ("Case 6 — Ardent SA",                                 2.86, 2.99, 3.32, 60.1),
    ],
)
def test_scoring_formula(case_label, demand, conversion, delivery, expected_overall):
    category_avgs = {
        "Demand":     demand,
        "Conversion": conversion,
        "Delivery":   delivery,
    }

    cat_scores, overall = compute_scores(CATEGORIES, category_avgs)
    results = {row["category"]: row for row in cat_scores}

    # category avg pass-through
    assert results["Demand"]["avg"]     == demand
    assert results["Conversion"]["avg"] == conversion
    assert results["Delivery"]["avg"]   == delivery

    # overall score
    assert round(overall, 1) == expected_overall, (
        f"{case_label}: expected overall={expected_overall}, got {round(overall,1)}"
    )
if __name__ == "__main__":
    cases = [
        ("Case 1 — Strong Demand, Weak Conversion, Mid Delivery", 4.5, 2.0, 3.0),
        ("Case 2 — Weak Demand, Strong Conversion, Mid Delivery", 2.0, 4.5, 3.0),
        ("Case 3 — All Strong",                                   4.5, 4.5, 4.5),
        ("Case 4 — All Weak",                                     1.5, 1.5, 1.5),
        ("Case 5 — Mid Demand, Mid Conversion, Weak Delivery",    3.0, 3.0, 2.0),
        ("Case 6 — Ardent SA",                                2.86, 2.99, 3.32),
    ]

    print(f"\n{'Case':<50} {'Demand':>8} {'Conv':>8} {'Delivery':>10} {'Overall':>9}")
    print("-" * 88)
    for label, d, c, dv in cases:
        avgs = {"Demand": d, "Conversion": c, "Delivery": dv}
        _, overall = compute_scores(CATEGORIES, avgs)
        print(f"{label:<50} {d:>8.2f} {c:>8.2f} {dv:>10.2f} {round(overall,1):>9.1f}")

Overwriting test_scoring.py


In [ ]:
# Install pytest
!pip install pytest -q

In [ ]:
# Run the tests
!python -m pytest test_scoring.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: typeguard-4.5.2, anyio-4.13.0, langsmith-0.8.12
collected 6 items                                                              

test_scoring.py::test_scoring_formula[Case 1 \u2014 Strong Demand, Weak Conversion, Mid Delivery-4.5-2.0-3.0-64.0] PASSED [ 16%]
test_scoring.py::test_scoring_formula[Case 2 \u2014 Weak Demand, Strong Conversion, Mid Delivery-2.0-4.5-3.0-64.0] PASSED [ 33%]
test_scoring.py::test_scoring_formula[Case 3 \u2014 All Strong-4.5-4.5-4.5-90.0] PASSED [ 50%]
test_scoring.py::test_scoring_formula[Case 4 \u2014 All Weak-1.5-1.5-1.5-30.0] PASSED [ 66%]
test_scoring.py::test_scoring_formula[Case 5 \u2014 Mid Demand, Mid Conversion, Weak Delivery-3.0-3.0-2.0-56.0] PASSED [ 83%]
test_scoring.py::test_scoring_formula[Case 6 \u2014 Ardent SA-2.86-2.99-3.32-60.1] PAS

#### Baseline Scores for each case. This is the "before" column for the Week 3 comparison

In [ ]:
!python test_scoring.py


Case                                                 Demand     Conv   Delivery   Overall
----------------------------------------------------------------------------------------
Case 1 — Strong Demand, Weak Conversion, Mid Delivery     4.50     2.00       3.00      64.0
Case 2 — Weak Demand, Strong Conversion, Mid Delivery     2.00     4.50       3.00      64.0
Case 3 — All Strong                                    4.50     4.50       4.50      90.0
Case 4 — All Weak                                      1.50     1.50       1.50      30.0
Case 5 — Mid Demand, Mid Conversion, Weak Delivery     3.00     3.00       2.00      56.0
Case 6 — Ardent SA                                     2.86     2.99       3.32      60.1


### Write tests/test_engine.py stubs — placeholder tests for the recommendation engine